# Looking at LANDFIRE data

Trying to figure out what was downloaded and how to see the maps. My LANDFIRE download is for the east, so I am keeping it separate from the western fire model.

## 1. Import stuff

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Find the main project folder so the data paths still work from explorations/
PROJECT_ROOT = next((p for base in (Path.cwd(), Path.cwd().resolve()) for p in [base, *base.parents] if (p / "data" / "raw").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find data/raw. Put explorations/ inside your fire-prediction project.")
RAW = PROJECT_ROOT / "data" / "raw"
print("Project root:", PROJECT_ROOT)
print("Data folder:", RAW)
import zipfile
import numpy as np
import rasterio
from rasterio.enums import Resampling

## 2. See what files I downloaded

In [ ]:
landfire_dir = RAW / 'landfire_east'
landfire_files = sorted(p for p in landfire_dir.rglob('*') if p.is_file()) if landfire_dir.exists() else []
display(pd.DataFrame([{'file':str(p.relative_to(RAW)), 'MB':round(p.stat().st_size/1e6,2)} for p in landfire_files]))
archives = [p for p in [RAW / 'landfire_east.zip', *landfire_files] if p.is_file() and p.suffix.lower()=='.zip']
for archive in archives:
    print('\nZIP:', archive.relative_to(RAW))
    with zipfile.ZipFile(archive) as z:
        for name in z.namelist()[:30]: print('  ', name)
        if len(z.namelist())>30:print(f' ...and {len(z.namelist())-30} more entries')

## 3. Find the map/raster files

In [ ]:
extensions={'.tif','.tiff','.img','.vrt'}
rasters=[p for p in landfire_files if p.suffix.lower() in extensions]
print('Detected rasters:', len(rasters))
for i,p in enumerate(rasters): print(i, p.relative_to(RAW))
if not rasters:
    print('No extracted raster found. The downloaded files may be zipped or use another format.')

## 4. Check one raster file

In [ ]:
if rasters:
    raster_path=rasters[0]  # try a different number for another file
    with rasterio.open(raster_path) as src:
        print('File:', raster_path.relative_to(RAW))
        print('CRS:',src.crs,'bounds:',src.bounds)
        print('Pixels:',src.width,'x',src.height,'bands:',src.count)
        print('Nodata:',src.nodata,'dtype:',src.dtypes)
        print('Descriptions:',src.descriptions)
        print('Tags:',src.tags())
else:
    print('Nothing to open yet.')

## 5. Display the raster

In [ ]:
if rasters:
    with rasterio.open(raster_path) as src:
        scale=min(1,900/max(src.width,src.height))
        h=max(1,int(src.height*scale));w=max(1,int(src.width*scale))
        # Keep the fuel category numbers as they are
        image=src.read(1,out_shape=(h,w),resampling=Resampling.nearest,masked=True)
        fig,ax=plt.subplots(figsize=(11,7))
        im=ax.imshow(image,interpolation='nearest')
        ax.set_title(raster_path.name)
        plt.colorbar(im,ax=ax,shrink=.7)
        plt.show()
else:
    print('No raster detected. Check the archive names above.')

## 6. Look at common values

In [ ]:
if rasters:
    values=image.compressed() if np.ma.isMaskedArray(image) else image.reshape(-1)
    counts=pd.Series(values).value_counts().head(20)
    display(counts.rename_axis('raster_value').rename('pixels_in_preview').to_frame())
    print('Use the LANDFIRE product legend to interpret category codes.')

## Notes

Some numbers in these rasters stand for categories (like fuel types), while others are regular values (like elevation). I need to check what each layer means before using it in a model.